# 🤖 Notebook 3: Traditional Machine Learning Pipeline

**Môn học:** Học Máy (CO3001) — HK I (2025-2026)  
**Nhóm:** Hoàng Xuân Bách · Nguyễn Việt Hùng · Trần Văn Hùng

---
### Mục tiêu
- Train & đánh giá các mô hình ML truyền thống
- So sánh hiệu suất giữa các model
- Feature importance & learning curves
- Hyperparameter tuning (Bonus)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib

warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from modules import config
from modules.models import ModelTrainer
from modules.evaluation import ModelEvaluator
from modules.features import FeatureEngineer

print('✅ Libraries loaded!')

## 2. Load Preprocessed Data

In [ ]:
X_train = np.load(config.FEATURES_DIR / 'X_train.npy')
X_test  = np.load(config.FEATURES_DIR / 'X_test.npy')
y_train = np.load(config.FEATURES_DIR / 'y_train.npy')
y_test  = np.load(config.FEATURES_DIR / 'y_test.npy')

with open(config.FEATURES_DIR / 'feature_names.txt') as f:
    feature_names = [l.strip() for l in f]

print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'Features: {len(feature_names)}')

## 3. Feature Engineering (Optional)

### 3.1 Loại Bỏ Features Tương Quan Cao

In [ ]:
engineer = FeatureEngineer(verbose=True)

X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df  = pd.DataFrame(X_test,  columns=feature_names)

X_train_reduced, removed = engineer.remove_correlated_features(
    X_train_df, threshold=config.CORRELATION_THRESHOLD
)
X_test_reduced = X_test_df.drop(columns=removed, errors='ignore')

print(f'\nFeatures còn lại: {X_train_reduced.shape[1]} (đã bỏ {len(removed)})')

# Dùng data sau khi giảm chiều
X_tr = X_train_reduced.values
X_te = X_test_reduced.values
feat_names = X_train_reduced.columns.tolist()

### 3.2 PCA (Thử nghiệm các mức variance)

In [ ]:
pca_results = {}
for vr in config.PCA_VARIANCE_RATIOS:
    X_pca, n_comp = engineer.apply_pca(X_tr, variance_ratio=vr, fit=True)
    pca_results[vr] = n_comp
    print(f'Variance {vr*100:.0f}%: {X_tr.shape[1]} → {n_comp} components')

# Không áp dụng PCA nếu số chiều đã nhỏ
# Uncomment để dùng PCA:
# X_tr, _ = engineer.apply_pca(X_tr, variance_ratio=0.95, fit=True)
# X_te, _ = engineer.apply_pca(X_te, fit=False)

## 4. Train Tất Cả Models

In [ ]:
trainer = ModelTrainer(verbose=True)

print('=== Bắt đầu training các models ===')
training_results = trainer.train_all_models(X_tr, y_train)

print(f'\n✅ Đã train {len(trainer.models)} models!')

## 5. Đánh Giá Models

In [ ]:
evaluator = ModelEvaluator(verbose=False)  # tắt verbose để gọn

results_df = evaluator.evaluate_multiple_models(trainer.models, X_te, y_test)

print('\n=== Model Comparison ===')
results_df.sort_values('accuracy', ascending=False).round(4)

## 6. Visualizations

### 6.1 Model Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrics = ['accuracy', 'f1']
colors  = ['steelblue', 'coral']

for ax, metric, color in zip(axes, metrics, colors):
    sorted_df = results_df[metric].sort_values()
    bars = ax.barh(sorted_df.index, sorted_df.values, color=color, edgecolor='white', alpha=0.85)
    ax.set_xlabel(metric.capitalize(), fontsize=12)
    ax.set_title(f'Models by {metric.upper()}', fontsize=13, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.axvline(sorted_df.values[-1], linestyle='--', color='black', alpha=0.4, label='Best')
    for bar, val in zip(bars, sorted_df.values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Traditional ML Models — Performance Comparison',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Confusion Matrix — Best Model

In [ ]:
best_name  = results_df['accuracy'].idxmax()
best_model = trainer.models[best_name]
y_pred     = best_model.predict(X_te)

print(f'Best model: {best_name}')
print(f'Accuracy  : {results_df.loc[best_name, "accuracy"]:.4f}')

evaluator.plot_confusion_matrix(
    y_test, y_pred,
    title=f'Confusion Matrix — {best_name}',
    save_path=config.get_figure_path(f'confusion_matrix_{best_name}')
)

### 6.3 ROC Curve

In [ ]:
if len(np.unique(y_test)) == 2 and hasattr(best_model, 'predict_proba'):
    y_proba = best_model.predict_proba(X_te)[:, 1]
    evaluator.plot_roc_curve(
        y_test, y_proba,
        title=f'ROC Curve — {best_name}',
        save_path=config.get_figure_path(f'roc_curve_{best_name}')
    )
else:
    print('Multi-class hoặc model không có predict_proba – bỏ qua ROC curve.')

### 6.4 Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    evaluator.plot_feature_importance(
        best_model, feat_names,
        top_n=min(20, len(feat_names)),
        title=f'Feature Importance — {best_name}',
        save_path=config.get_figure_path(f'feature_importance_{best_name}')
    )
else:
    print(f'{best_name} không có feature_importances_.')

### 6.5 Learning Curve

In [ ]:
evaluator.plot_learning_curve(
    best_model, X_tr, y_train,
    cv=config.CV_FOLDS,
    title=f'Learning Curve — {best_name}',
    save_path=config.get_figure_path(f'learning_curve_{best_name}')
)

### 6.6 Classification Report

In [ ]:
evaluator.generate_classification_report(y_test, y_pred)

## 7. Hyperparameter Tuning (Bonus)

In [ ]:
# Tune RandomForest với GridSearchCV
if 'RandomForest' in trainer.models:
    print('Tuning Random Forest...')
    best_rf, best_params, best_score = trainer.hyperparameter_tuning(
        trainer.models['RandomForest'],
        X_tr, y_train,
        param_grid=config.PARAM_GRIDS['RandomForest'],
        search_type='grid',
        cv=config.CV_FOLDS
    )
    
    tuned_metrics = evaluator.evaluate_model(best_rf, X_te, y_test, 'RF_Tuned')
    orig_metrics  = evaluator.evaluate_model(trainer.models['RandomForest'],
                                             X_te, y_test, 'RF_Original')
    print(f'\nOriginal RF accuracy: {orig_metrics["accuracy"]:.4f}')
    print(f'Tuned    RF accuracy: {tuned_metrics["accuracy"]:.4f}')
else:
    print('RandomForest không có trong trainer.models')

## 8. Lưu Best Model & Kết Quả

In [ ]:
# Lưu best model
trainer.save_model(best_model, f'best_model_{best_name}')

# Lưu kết quả
results_df.to_csv(config.REPORTS_DIR / 'model_results.csv')
evaluator.save_results(results_df)

print(f'\n✅ Đã lưu model: best_model_{best_name}.pkl')
print(f'✅ Đã lưu kết quả: reports/model_results.csv')
print(f'\n=== Best Model Summary ===')
print(f'Model   : {best_name}')
for m, v in results_df.loc[best_name].items():
    print(f'{m:12s}: {v:.4f}')